# typedframes + Jupyter notebooks

`typedframes check` reads `.ipynb` files directly -- no conversion step needed.
Errors are reported as `notebook.ipynb:cell N:line:col`, mapped back to the cell
that actually produced them, rather than a line number in the raw JSON.

Run it from this directory:

```shell
uv run typedframes check ipynb_example.ipynb
```

This notebook has one intentional bug, in the last cell, to show what gets caught.

In [1]:
from typing import Annotated

import pandas as pd

from typedframes import BaseSchema, Column


class Orders(BaseSchema):
    order_id = Column(type=int)
    customer_id = Column(type=int)
    amount = Column(type=float)


orders: Annotated[pd.DataFrame, Orders] = pd.DataFrame(
    {
        "order_id": [1, 2, 3],
        "customer_id": [10, 20, 30],
        "amount": [9.99, 19.99, 29.99],
    }
)
print(orders)

   order_id  customer_id  amount
0         1           10    9.99
1         2           20   19.99
2         3           30   29.99


IPython magics are tolerated: the checker blanks the magic line in place and keeps
checking the rest of the cell.

In [2]:
%matplotlib inline

total = orders[Orders.amount.s].sum()
print(f"Total revenue: {total:.2f}")

Total revenue: 59.97


The cell below intentionally accesses a column that doesn't exist (`revenue` --
the real column is `amount`). `typedframes check` catches this at lint time;
actually running the cell would raise a `KeyError` instead, so it's left unrun.

In [ ]:
orders["revenue"]  # real column is "amount" -- typedframes catches this without running anything